# Carrega os dados da silver

In [ ]:
# Conecta o colab ao drive
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Importa as bibliotecas necessárias
import pyarrow.dataset as ds
import pandas as pd
import numpy as np
import os
import shutil

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/dadosfera/amazon_products_silver"

df = pd.read_parquet(path, engine="pyarrow")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1379629 entries, 0 to 1379628
Data columns (total 17 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   product_id             1379629 non-null  string 
 1   product_title          1379629 non-null  string 
 2   category_id            1379629 non-null  int32  
 3   category_name          1379629 non-null  string 
 4   price                  1379629 non-null  float32
 5   list_price             1379629 non-null  float32
 6   rating                 1379629 non-null  float32
 7   review_count           1379629 non-null  int32  
 8   has_rating             1379629 non-null  bool   
 9   weighted_score         1379629 non-null  float32
 10  units_sold_last_month  1379629 non-null  int32  
 11  is_best_seller         1379629 non-null  bool   
 12  has_title              1379629 non-null  bool   
 13  image_url              1379629 non-null  string 
 14  product_url       

In [ ]:
df.head()

,product_id,product_title,category_id,category_name,price,list_price,rating,review_count,has_rating,weighted_score,units_sold_last_month,is_best_seller,has_title,image_url,product_url,price_segment,popularity_tier
0,B01M3RYHP0,Men's Tag-Free Cotton Briefs,110,Men's Clothing,8.38,22.99,4.6,0,True,0.0,4000,True,True,https://m.media-amazon.com/images/I/61Qz7eM36p...,https://www.amazon.com/dp/B01M3RYHP0,budget,high
1,B07P7M18C6,Energy Unisex Easy-On/Easy-Off Knee High Compr...,110,Men's Clothing,11.88,0.00,4.3,0,True,0.0,5000,False,True,https://m.media-amazon.com/images/I/8146gI8s+S...,https://www.amazon.com/dp/B07P7M18C6,budget,high
2,B0B62HGK2H,Hooded Rain Poncho Waterproof Raincoat Jacket ...,110,Men's Clothing,9.99,13.99,4.5,0,True,0.0,2000,False,True,https://m.media-amazon.com/images/I/61JNcVZCPF...,https://www.amazon.com/dp/B0B62HGK2H,budget,high
3,B01L8JJ57U,"Men's EcoSmart Fleece Sweatshirt, Cotton-Blend...",110,Men's Clothing,10.99,18.00,4.6,0,True,0.0,3000,False,True,https://m.media-amazon.com/images/I/71fyw1U54G...,https://www.amazon.com/dp/B01L8JJ57U,budget,high
4,B08MBFGL13,"Men's Max Cushioned Crew Socks, Moisture-Wicki...",110,Men's Clothing,11.00,0.00,4.6,0,True,0.0,4000,False,True,https://m.media-amazon.com/images/I/81vTEPIns+...,https://www.amazon.com/dp/B08MBFGL13,budget,high


# Camada gold (métricas e agregações)

## Métricas por categoria

In [ ]:
gold_category_metrics = (
    df
    .groupby(["category_id", "category_name"], as_index=False)
    .agg(
        total_products=("product_id", "count"),
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        total_units_sold_last_month=("units_sold_last_month", "sum"),
        pct_best_seller=("is_best_seller", "mean"),
        total_reviews=("review_count", "sum"),
        avg_weighted_score=("weighted_score", "mean"),
    )
)

### Rating médio só para produtos com rating

In [ ]:
rating_only = (
    df[df["has_rating"]]
    .groupby(["category_id"], as_index=False)
    .agg(avg_rating=("rating", "mean"))
)

gold_category_metrics = gold_category_metrics.merge(rating_only, on="category_id", how="left")

gold_category_metrics["pct_best_seller"] = (gold_category_metrics["pct_best_seller"] * 100).astype("float32")
gold_category_metrics["avg_price"] = gold_category_metrics["avg_price"].astype("float32")
gold_category_metrics["median_price"] = gold_category_metrics["median_price"].astype("float32")
gold_category_metrics["avg_weighted_score"] = gold_category_metrics["avg_weighted_score"].astype("float32")
gold_category_metrics["avg_rating"] = gold_category_metrics["avg_rating"].astype("float32")

### Segmento mais comum por categoria

In [ ]:
dominant_seg = (
    df
    .groupby(["category_id", "price_segment"])
    .size()
    .reset_index(name="cnt")
    .sort_values(["category_id", "cnt"], ascending=[True, False])
    .drop_duplicates("category_id")
    .rename(columns={"price_segment": "dominant_price_segment"})
    [["category_id", "dominant_price_segment"]]
)

gold_category_metrics = gold_category_metrics.merge(dominant_seg, on="category_id", how="left")
gold_category_metrics["dominant_price_segment"] = gold_category_metrics["dominant_price_segment"].astype("string")

In [ ]:
gold_category_metrics.head()

,category_id,category_name,total_products,avg_price,median_price,total_units_sold_last_month,pct_best_seller,total_reviews,avg_weighted_score,avg_rating,dominant_price_segment
0,1,Beading & Jewelry Making,8229,14.034677,9.99,1098150,0.315956,473440,4.966172,4.490352,budget
1,2,Fabric Decorating,2005,18.262779,11.99,64600,0.149626,588603,11.240300,4.326713,budget
2,3,Knitting & Crochet Supplies,8143,17.136339,12.99,260700,0.147366,1721220,9.620075,4.481169,budget
3,4,Printmaking Supplies,5347,54.780857,22.26,144550,0.205723,767332,11.302085,4.320240,luxury
4,5,Scrapbooking & Stamping Supplies,5886,14.912978,9.99,498550,0.373768,0,0.000000,4.523577,budget


## Série temporal sintética

In [ ]:
base_seg = (
    df.groupby(["category_id", "category_name", "price_segment", "popularity_tier"], as_index=False)
    .agg(
        anchor_units=("units_sold_last_month", "sum"),
        median_price=("price", "median"),
        avg_price=("price", "mean"),
    )
)

mes_ancora = pd.Timestamp("2023-12-01")
meses = pd.date_range(end=mes_ancora, periods=12, freq="MS")

### Define os períodos com sazonalidade

In [ ]:
sazonalidade_mensal = {1:1.05,2:0.90,3:0.95,4:0.98,5:1.00,6:1.02,7:1.03,8:1.02,9:1.00,10:1.02,11:1.20,12:1.35}
sazonalidade = np.array([sazonalidade_mensal[m.month] for m in meses], dtype="float32")
sazonalidade_ancora = sazonalidade_mensal[mes_ancora.month]

rng = np.random.default_rng(42)
t = np.arange(len(meses)) - (len(meses) - 1)

rows = []
for _, r in base_seg.iterrows():
    anchor_units = int(r["anchor_units"])
    if anchor_units < 10:  # evita combinações mortas
        continue

    trend = rng.uniform(0.98, 1.02)
    noise = rng.normal(0.0, 0.08, size=len(meses))

    units = anchor_units * (sazonalidade / sazonalidade_ancora) * (trend ** t) * (1 + noise)
    units = np.clip(np.round(units), 0, None).astype("int32")

    med_price = float(r["median_price"])
    revenue = (units.astype("float32") * med_price).astype("float32")

    for i, m in enumerate(meses):
        rows.append({
            "month": m,
            "category_id": int(r["category_id"]),
            "category_name": r["category_name"],
            "price_segment": r["price_segment"],
            "popularity_tier": r["popularity_tier"],
            "units_sold": int(units[i]),
            "revenue": float(revenue[i]),
            "median_price": float(med_price),
            "avg_price": float(r["avg_price"]),
        })

gold_category_segment_tier_monthly = pd.DataFrame(rows)

### Verificação de qualidade

In [ ]:
assert gold_category_segment_tier_monthly["units_sold"].ge(0).all()
assert gold_category_segment_tier_monthly["month"].nunique() == 12

In [ ]:
gold_category_segment_tier_monthly.head()

,month,category_id,category_name,price_segment,popularity_tier,units_sold,revenue,median_price,avg_price
0,2023-01-01,1,Beading & Jewelry Making,budget,high,454235,4038149.25,8.89,8.535537
1,2023-02-01,1,Beading & Jewelry Making,budget,high,455106,4045892.50,8.89,8.535537
2,2023-03-01,1,Beading & Jewelry Making,budget,high,492622,4379409.50,8.89,8.535537
3,2023-04-01,1,Beading & Jewelry Making,budget,high,403220,3584626.00,8.89,8.535537
4,2023-05-01,1,Beading & Jewelry Making,budget,high,441542,3925308.50,8.89,8.535537


## Produtos selecionados

In [ ]:
gold_products_curated = df.copy()

# Normalizações dentro da categoria
gold_products_curated["units_norm"] = (
    gold_products_curated.groupby("category_id")["units_sold_last_month"]
    .transform(lambda s: (s - s.min()) / (s.max() - s.min() + 1e-9))
).astype("float32")

gold_products_curated["wscore_norm"] = (
    gold_products_curated.groupby("category_id")["weighted_score"]
    .transform(lambda s: (s - s.min()) / (s.max() - s.min() + 1e-9))
).astype("float32")

gold_products_curated["strategic_score"] = (
    0.55 * gold_products_curated["wscore_norm"] +
    0.40 * gold_products_curated["units_norm"] +
    0.05 * gold_products_curated["is_best_seller"].astype("int8")
).astype("float32")

### Rank e flag dos produtos

In [ ]:
# Criando rank dentro da categoria
gold_products_curated["category_rank"] = (
    gold_products_curated
    .groupby("category_id")["strategic_score"]
    .rank(method="first", ascending=False)
    .astype("int32")
)

In [ ]:
# Flag de melhores produtos
gold_products_curated["is_top_10_category"] = gold_products_curated["category_rank"] <= 10

In [ ]:
gold_products_curated_to_save = gold_products_curated.drop(columns=["units_norm", "wscore_norm"])

In [ ]:
gold_products_curated_to_save.head()

,product_id,product_title,category_id,category_name,price,list_price,rating,review_count,has_rating,weighted_score,units_sold_last_month,is_best_seller,has_title,image_url,product_url,price_segment,popularity_tier,strategic_score,category_rank,is_top_10_category
0,B01M3RYHP0,Men's Tag-Free Cotton Briefs,110,Men's Clothing,8.38,22.99,4.6,0,True,0.0,4000,True,True,https://m.media-amazon.com/images/I/61Qz7eM36p...,https://www.amazon.com/dp/B01M3RYHP0,budget,high,0.21,748,False
1,B07P7M18C6,Energy Unisex Easy-On/Easy-Off Knee High Compr...,110,Men's Clothing,11.88,0.00,4.3,0,True,0.0,5000,False,True,https://m.media-amazon.com/images/I/8146gI8s+S...,https://www.amazon.com/dp/B07P7M18C6,budget,high,0.20,796,False
2,B0B62HGK2H,Hooded Rain Poncho Waterproof Raincoat Jacket ...,110,Men's Clothing,9.99,13.99,4.5,0,True,0.0,2000,False,True,https://m.media-amazon.com/images/I/61JNcVZCPF...,https://www.amazon.com/dp/B0B62HGK2H,budget,high,0.08,1144,False
3,B01L8JJ57U,"Men's EcoSmart Fleece Sweatshirt, Cotton-Blend...",110,Men's Clothing,10.99,18.00,4.6,0,True,0.0,3000,False,True,https://m.media-amazon.com/images/I/71fyw1U54G...,https://www.amazon.com/dp/B01L8JJ57U,budget,high,0.12,1034,False
4,B08MBFGL13,"Men's Max Cushioned Crew Socks, Moisture-Wicki...",110,Men's Clothing,11.00,0.00,4.6,0,True,0.0,4000,False,True,https://m.media-amazon.com/images/I/81vTEPIns+...,https://www.amazon.com/dp/B08MBFGL13,budget,high,0.16,947,False


## Top 100 produtos

In [ ]:
gold_top_products = gold_products_curated[gold_products_curated["category_rank"] <= 100].copy()

In [ ]:
gold_top_products.head()

,product_id,product_title,category_id,category_name,price,list_price,rating,review_count,has_rating,weighted_score,...,has_title,image_url,product_url,price_segment,popularity_tier,units_norm,wscore_norm,strategic_score,category_rank,is_top_10_category
480,B0108L0QHM,EnviroCare Replacement Micro Filtration Vacuum...,175,Vacuum Cleaners & Floor Care,9.99,0.00,4.8,7615,True,42.902431,...,True,https://m.media-amazon.com/images/I/81zQZqw9to...,https://www.amazon.com/dp/B0108L0QHM,budget,high,0.500000,0.852176,0.668697,10,True
486,B06Y3S2JQT,KEEPOW Style 7/9/10 P/N 3031120 Replacement Be...,175,Vacuum Cleaners & Floor Care,9.99,11.99,4.5,10482,True,41.658794,...,True,https://m.media-amazon.com/images/I/71cSgBxl4O...,https://www.amazon.com/dp/B06Y3S2JQT,budget,high,0.333333,0.827474,0.638444,15,False
487,B00OGKYNF8,Shark Replacement Filter Set XFF350 Navigator ...,175,Vacuum Cleaners & Floor Care,7.11,0.00,4.6,4943,True,39.127277,...,True,https://m.media-amazon.com/images/I/41W4Il2u52...,https://www.amazon.com/dp/B00OGKYNF8,budget,high,0.333333,0.777190,0.560788,38,False
488,B01HID4UZ2,"Bissell Crosswave Wood Floor Cleaning Formula,...",175,Vacuum Cleaners & Floor Care,9.99,0.00,4.7,4899,True,39.935856,...,True,https://m.media-amazon.com/images/I/71lelVO9gs...,https://www.amazon.com/dp/B01HID4UZ2,budget,high,0.333333,0.793251,0.569621,35,False
489,B004ND78GE,"BLACK+DECKER Hand Vacuum Filter, Washable, Rep...",175,Vacuum Cleaners & Floor Care,7.52,0.00,4.6,11082,True,42.840572,...,True,https://m.media-amazon.com/images/I/41LmekLiAa...,https://www.amazon.com/dp/B004ND78GE,budget,high,0.333333,0.850948,0.601355,23,False


# Salva os dataframes

In [ ]:
path_gold_category_metrics = "/content/drive/MyDrive/Colab Notebooks/dadosfera/gold_category_metrics"

if os.path.exists(path_gold_category_metrics):
    shutil.rmtree(path_gold_category_metrics)

gold_category_metrics.to_parquet(path_gold_category_metrics, engine="pyarrow", index=False)

In [ ]:
path_gold_category_segment_tier_monthly = "/content/drive/MyDrive/Colab Notebooks/dadosfera/gold_category_segment_tier_monthly"

if os.path.exists(path_gold_category_segment_tier_monthly):
    shutil.rmtree(path_gold_category_segment_tier_monthly)

gold_category_segment_tier_monthly.to_parquet(path_gold_category_segment_tier_monthly, engine="pyarrow", index=False)

In [ ]:
path_gold_products_curated = "/content/drive/MyDrive/Colab Notebooks/dadosfera/gold_products_curated"

if os.path.exists(path_gold_products_curated):
    shutil.rmtree(path_gold_products_curated)

gold_products_curated_to_save.to_parquet(path_gold_products_curated, engine="pyarrow", index=False)

In [ ]:
path_gold_top_products = "/content/drive/MyDrive/Colab Notebooks/dadosfera/gold_top_products"

if os.path.exists(path_gold_top_products):
    shutil.rmtree(path_gold_top_products)

gold_top_products.to_parquet(path_gold_top_products, engine="pyarrow", index=False)